# Phase 5: Product Family Discovery

This notebook completes the roadmap's Phase 5 tasks:

25. Build a station presence matrix.
26. Cluster products using KMeans, DBSCAN, and Hierarchical Clustering.
27. Identify product families.
28. Compare failure rates across product families.
29. Train separate models for each family.

The station presence matrix is built from the raw `train_date.csv` and `test_date.csv` files. The target comes from raw `train_numeric.csv`. Family-specific models use the Phase 4 engineered feature table because that table contains the timing, delay, station-count, complexity, and line-level features created in the previous phase.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

## Run The Phase 5 Pipeline

The Python script below is the source of truth for this phase. It creates the raw station-presence matrices, clusters train/test products, writes product-family reports, and trains one model per final product family. If the station-presence matrices already exist, the script reuses them to avoid rescanning the large raw date files.

In [ ]:
import subprocess

script_path = PROJECT_ROOT / 'src' / 'data' / 'phase5_product_family_discovery.py'
subprocess.run([sys.executable, str(script_path)], cwd=PROJECT_ROOT, check=True)

## Load Phase 5 Outputs

These output tables let us inspect the family definitions, clustering behavior, failure rates, and model quality without rereading the full raw datasets.

In [ ]:
import pandas as pd

processed_dir = PROJECT_ROOT / 'data' / 'processed'
reports_dir = PROJECT_ROOT / 'reports'

cluster_diagnostics = pd.read_csv(reports_dir / 'phase5_cluster_diagnostics.csv')
family_failure_rates = pd.read_csv(reports_dir / 'phase5_product_family_failure_rates.csv')
family_profiles = pd.read_csv(reports_dir / 'phase5_product_family_profiles.csv')
model_metrics = pd.read_csv(reports_dir / 'phase5_family_model_metrics.csv')

train_families = pd.read_csv(processed_dir / 'phase5_train_product_families.csv')
test_families = pd.read_csv(processed_dir / 'phase5_test_product_families.csv')

cluster_diagnostics

## Station Presence Matrix Check

Each `present_L*_S*` column is a binary indicator showing whether a product passed through that station at least once. This is the manufacturing-path representation used for clustering.

In [ ]:
train_presence_preview = pd.read_csv(processed_dir / 'phase5_train_station_presence_matrix.csv', nrows=5)
station_cols = [col for col in train_presence_preview.columns if col.startswith('present_')]

print(f'Train products labeled: {len(train_families):,}')
print(f'Test products labeled: {len(test_families):,}')
print(f'Station indicator columns: {len(station_cols):,}')
train_presence_preview.head()

## Product Family Failure Rates

The final product family is the KMeans family because KMeans assigns every train and test product to a family. DBSCAN and hierarchical labels are still saved as comparison clustering views.

In [ ]:
family_failure_rates.sort_values('failure_rate_pct', ascending=False)

## Family Profiles

The profile table adds the highest-presence stations for each family. These station lists make the clusters easier to interpret as manufacturing paths instead of abstract numeric labels.

In [ ]:
family_profiles[['final_product_family', 'part_count', 'failure_rate_pct', 'avg_station_count', 'avg_line_count', 'top_stations']]

## Family-Specific Models

Each final product family receives its own `HistGradientBoostingClassifier`. The models use balanced sample weights because Bosch failures are rare. Large families are stratified down to a maximum training sample so the notebook remains practical to rerun on a laptop.

In [ ]:
model_metrics.sort_values('roc_auc', ascending=False)

## Phase 5 Summary

This phase creates product-family labels that can be reused in later modeling work. The failure-rate report highlights which manufacturing paths carry elevated quality risk, and the family-specific models provide a first modeling baseline for each path group.